# IAM paired OCR augmentation benchmark
Run All defaults to generation only: SpiS-GAN, FW-GAN and HiGAN+ produce downloadable paired datasets plus prepared.zip. Set RUN_STAGE to 'ocr' for saved outputs without GAN source/checkpoints, or 'all' for generation then OCR. Internet and GPU acceleration are required. A Kaggle session is not guaranteed to finish the full OCR budget; save outputs and explicitly restore them next session. Default IAM32 comparison labels HiGAN+ native64→32 bilinear. Optional IAM64 is HiGAN+ only, not a three-model comparison. No quality filtering. Real test data never chooses checkpoints.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time, hashlib, urllib.request, tempfile, shutil
RUN_STAGE = 'generation'  # 'generation', 'ocr', 'all'
OUTPUT_DIR = Path('/kaggle/working/ocr_benchmark_outputs')
CODE_DIR = Path('/kaggle/working/ocr_benchmark_code')
RUNTIME_DIR = Path('/tmp/ocr_benchmark_runtime')
SOURCES_DIR = Path('/tmp/ocr_benchmark_sources')
DATA_DIR = Path('/kaggle/working/iam')  # Or an explicit read-only Kaggle input mount.
PREVIOUS_OUTPUT_DIR = None  # Explicit prior output root, never modified.
CHECKPOINT_PATHS = {'spis_gan': None, 'fw_gan': None}
COHORT_HEIGHT = 32  # 64 enables ONLY HiGAN+.
SESSION_HOURS = 10.5
GPU_SLOTS = ['0', '1']
WANDB_MODE = 'disabled'  # online/offline/disabled; key via Secrets/env only.
WANDB_PROJECT = 'iam-ocr-augmentation'
SESSION_STARTED = time.monotonic()
assert RUN_STAGE in {'generation','ocr','all'} and COHORT_HEIGHT in {32,64}
os.environ.update(HF_HOME='/tmp/ocr_benchmark_hf_cache', TOKENIZERS_PARALLELISM='false')
WORKER_ENV = dict(os.environ)



In [ ]:
def checked(argv, **kwargs):
    return subprocess.run([str(x) for x in argv], check=True, **kwargs)
if not CODE_DIR.exists():
    checked(['git','clone','--branch','ocr_benchmark','--single-branch',
             'https://github.com/rquq/HGGAN_test.git',CODE_DIR])
if not (CODE_DIR/'benchmark.py').is_file():
    raise RuntimeError('CODE_DIR is not the benchmark branch checkout')
# Never reset or overwrite an existing checkout.
code_head = subprocess.check_output(['git','-C',str(CODE_DIR),'rev-parse','HEAD'],text=True).strip()
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Benchmark source HEAD:', code_head)
requirements = CODE_DIR/'requirements.txt'
requirements_sha = hashlib.sha256(requirements.read_bytes()).hexdigest()
PYTHON = RUNTIME_DIR/'bin/python'
stamp = RUNTIME_DIR/'requirements.sha256'
if not PYTHON.exists():
    checked([sys.executable,'-m','venv',RUNTIME_DIR])
if not stamp.exists() or stamp.read_text().strip() != requirements_sha:
    checked([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',requirements],env=WORKER_ENV)
    stamp.write_text(requirements_sha+'\n')
# No system-site-packages; no environment/source/HF caches in Save Version output.
checked([PYTHON,'-c','import torch,transformers,h5py,wandb,yaml; print("Worker imports ready")'],env=WORKER_ENV)
if WANDB_MODE == 'online' and not WORKER_ENV.get('WANDB_API_KEY'):
    from kaggle_secrets import UserSecretsClient
    WORKER_ENV['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')



In [ ]:
# Read source YAML only in the isolated worker; notebook parent is stdlib.
config_path = CODE_DIR/'configs'/('benchmark_'+str(COHORT_HEIGHT)+'.yml')
raw_config = subprocess.check_output([str(PYTHON),'-c',
    'import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))',str(config_path)],text=True,env=WORKER_ENV)
runtime = json.loads(raw_config)
runtime['output_dir'] = str(OUTPUT_DIR.resolve())
runtime['wandb'] = {'mode':WANDB_MODE,'project':WANDB_PROJECT}
runtime['dataset']['train'] = str((DATA_DIR/('iam_train_'+str(COHORT_HEIGHT)+'.hdf5')).resolve())
runtime['dataset']['test'] = str((DATA_DIR/('iam_test_'+str(COHORT_HEIGHT)+'.hdf5')).resolve())
DATA_DIR.mkdir(parents=True,exist_ok=True)
for split in ['train','test']:
    target = Path(runtime['dataset'][split])
    if not target.exists():
        url = 'https://github.com/rquq/HGGAN_test/releases/download/data/'+target.name
        fd, temporary = tempfile.mkstemp(prefix='.'+target.name,dir=target.parent)
        os.close(fd)
        try:
            urllib.request.urlretrieve(url,temporary)
            checked([PYTHON,'-c','import sys; from pipeline.data import validate_iam; validate_iam(sys.argv[1])',
                     temporary],cwd=CODE_DIR,env=WORKER_ENV)
            os.replace(temporary,target)
        finally:
            if os.path.exists(temporary): os.unlink(temporary)
    else:
        checked([PYTHON,'-c','import sys; from pipeline.data import validate_iam; validate_iam(sys.argv[1])',
                 target],cwd=CODE_DIR,env=WORKER_ENV)



In [ ]:
model_configs = {}
source_records = {'benchmark_head':code_head,'requirements_sha256':requirements_sha,'generators':{}}
for name, relative in runtime['generation']['models'].items():
    raw = subprocess.check_output([str(PYTHON),'-c',
        'import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))',str(CODE_DIR/relative)],text=True,env=WORKER_ENV)
    model = json.loads(raw)
    if RUN_STAGE != 'ocr':
        checkout = SOURCES_DIR/name
        checkout.parent.mkdir(parents=True,exist_ok=True)
        if not checkout.exists():
            checked(['git','clone',model['source_url'],checkout])
            checked(['git','-C',checkout,'checkout','--detach',model['source_revision']])
        head = subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip()
        if head != model['source_revision']:
            raise RuntimeError('Existing source revision mismatch: '+name)
        source_root = checkout/'HiGAN+' if name=='higanplus' else checkout
        model['source_root'] = str(source_root)
        model['source_config'] = str(source_root/'configs'/Path(model['source_config']).name)
        if name=='higanplus':
            checkpoint = source_root/'pretrained/deploy_HiGAN+.pth'
        else:
            explicit = CHECKPOINT_PATHS.get(name)
            filename = 'bestIAM.pth' if name=='spis_gan' else 'FW-GAN.pth'
            candidates = [Path(explicit)] if explicit else sorted(Path('/kaggle/input').rglob(filename))
            if len(candidates)!=1 or not candidates[0].is_file():
                raise RuntimeError('Supply exactly one CHECKPOINT_PATHS entry or '+filename+' input: '+name)
            checkpoint = candidates[0]
        model['checkpoint'] = str(checkpoint.resolve())
        if not checkpoint.is_file():
            raise FileNotFoundError(checkpoint)
        source_records['generators'][name] = {'revision':head,'url':model['source_url']}
    # OCR mode never clones or checks GAN checkpoints; portable bundles carry provenance.
    model_path = OUTPUT_DIR/('runtime_'+name+'.json')
    model_path.write_text(json.dumps(model,indent=2))
    model_configs[name] = str(model_path)
runtime['generation']['models'] = model_configs
RUNTIME_CONFIG = OUTPUT_DIR/'runtime_config.json'
RUNTIME_CONFIG.write_text(json.dumps(runtime,indent=2))
(OUTPUT_DIR/'source_manifest.json').write_text(json.dumps(source_records,indent=2))



In [ ]:
base_argv = [str(PYTHON),str(CODE_DIR/'benchmark.py')]
prepare_argv = base_argv+['prepare','--config',str(RUNTIME_CONFIG)]
if PREVIOUS_OUTPUT_DIR is not None:
    prepare_argv += ['--previous-output-dir',str(PREVIOUS_OUTPUT_DIR)]
checked(prepare_argv,cwd=CODE_DIR,env=WORKER_ENV)



In [ ]:
# Stdout is streamed directly; subprocess completion, not a printed token, ends monitoring.
remaining = max(0.,SESSION_HOURS*3600-(time.monotonic()-SESSION_STARTED))
checked(base_argv+['run','--config',str(RUNTIME_CONFIG),'--stage',RUN_STAGE,
                  '--gpu-slots',','.join(GPU_SLOTS),'--time-budget-seconds',str(remaining)],
        cwd=CODE_DIR,env=WORKER_ENV)
# Generation-only does not enter any OCR cell.



In [ ]:
checked(base_argv+['report','--config',str(RUNTIME_CONFIG)],cwd=CODE_DIR,env=WORKER_ENV)
print('Save Version outputs:',OUTPUT_DIR)
print('Paused means incomplete; restore this output explicitly next session.')



In [ ]:
from IPython.display import FileLink, display
download_paths = sorted(OUTPUT_DIR.glob('datasets/*/dataset.zip'))
download_paths += [p for p in [OUTPUT_DIR/'prepared.zip',OUTPUT_DIR/'results.csv',OUTPUT_DIR/'results.json'] if p.is_file()]
for path in download_paths:
    print(path)
    display(FileLink(str(path)))



Default FP32 OCR: stage1 TrOCR-small, 10,000 optimizer updates, effective batch 32 (micro2×accum16), real validation selection, greedy decoding. One worker sees exactly one T4 and uses logical cuda:0. Two independent jobs, never DataParallel/distributed. Session budget is cooperative: a running model load or optimizer update can finish after deadline; queued jobs stop. Clean pauses are not results. Results include only completed full-real-test metrics; incomplete rows remain marked. Restore best/last checkpoints, datasets and prepared cohort through PREVIOUS_OUTPUT_DIR; no caches or old runtime are restored. Rebuild the pinned environment each fresh session. Online W&B uses WANDB_API_KEY from Secrets/environment; do not paste keys into cells.
